# AgentCore Evaluation Pipeline Walkthrough

This notebook walks through the key components of an AgentCore CI/CD evaluation pipeline with MCP server role-based access control.

## Prerequisites

- AWS account with AgentCore access
- CDK stack deployed (see README.md)
- `pip install boto3 requests bedrock-agentcore-starter-toolkit`

In [ ]:
import os
import json
import time
import uuid
import urllib.parse
import requests as http_requests

# Configuration — update these from your CDK outputs
REGION = os.getenv("AWS_REGION", "ap-southeast-2")
AGENT_RUNTIME_ARN = os.getenv("AGENT_RUNTIME_ARN", "<your-agent-arn>")
AGENT_RUNTIME_ID = os.getenv("AGENT_RUNTIME_ID", "<your-agent-id>")
TOKEN_ENDPOINT = os.getenv("TOKEN_ENDPOINT", "<your-token-endpoint>")
OAUTH_CLIENT_ID = os.getenv("OAUTH_CLIENT_ID", "<your-client-id>")
OAUTH_CLIENT_SECRET = os.getenv("OAUTH_CLIENT_SECRET", "<your-client-secret>")
EVAL_THRESHOLD = float(os.getenv("EVAL_THRESHOLD", "0.8"))

## Step 1: Get OAuth Token

AgentCore runtimes with JWT auth require a Bearer token. CI pipelines use the `client_credentials` grant.

In [ ]:
def get_m2m_token():
    """Get M2M token via client_credentials grant."""
    resp = http_requests.post(
        TOKEN_ENDPOINT,
        data={
            "grant_type": "client_credentials",
            "client_id": OAUTH_CLIENT_ID,
            "client_secret": OAUTH_CLIENT_SECRET,
            "scope": "agentcore/invoke",
        },
    )
    resp.raise_for_status()
    return resp.json()["access_token"]

token = get_m2m_token()
print(f"Token acquired (first 20 chars): {token[:20]}...")

## Step 2: Invoke the Agent

OAuth-protected runtimes must be invoked via HTTPS with a Bearer token — the boto3 SDK doesn't support OAuth invocations.

In [ ]:
def invoke_agent(prompt, session_id, token):
    """Invoke AgentCore Runtime via HTTPS with Bearer token."""
    escaped_arn = urllib.parse.quote(AGENT_RUNTIME_ARN, safe="")
    url = f"https://bedrock-agentcore.{REGION}.amazonaws.com/runtimes/{escaped_arn}/invocations?qualifier=DEFAULT"

    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
        "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id,
    }

    resp = http_requests.post(url, headers=headers, data=json.dumps({"prompt": prompt}))
    resp.raise_for_status()
    return resp.json()

# Test prompts covering different tool types
prompts = [
    "How much is 2+2?",                                    # calculator (built-in)
    "How is the weather in Sydney?",                       # weather (built-in)
    "What is the capital of the US?",                      # get_capital_city (MCP, public)
    "What is the stock price of AAPL?",                    # get_stock_price (MCP, FinanceUser)
    "How many employees are in engineering?",              # get_employee_count (MCP, HRUser)
]

session_id = str(uuid.uuid4())
print(f"Session ID: {session_id}\n")

for prompt in prompts:
    print(f"Q: {prompt}")
    try:
        result = invoke_agent(prompt, session_id, token)
        print(f"A: {result}\n")
    except Exception as e:
        print(f"Error: {e}\n")

## Step 3: Run Evaluations

The `bedrock-agentcore-starter-toolkit` provides a high-level `Evaluation` class that handles trace collection and scoring.

In [ ]:
from bedrock_agentcore_starter_toolkit import Evaluation

# Wait for traces to propagate
print("Waiting 30s for trace propagation...")
time.sleep(30)

evaluators = [
    "Builtin.GoalSuccessRate",
    "Builtin.Correctness",
    "Builtin.ToolSelectionAccuracy",
    "Builtin.ToolParameterAccuracy",
]

results = Evaluation(region=REGION).run(
    agent_id=AGENT_RUNTIME_ID,
    session_id=session_id,
    evaluators=evaluators,
    output="eval_output.json",
)

# Display results
scores = {}
for r in results.results:
    if r.value is not None:
        if r.evaluator_name not in scores or r.value > scores[r.evaluator_name][0]:
            scores[r.evaluator_name] = (r.value, r.label)

print(f"\n{'─' * 50}")
print(f"{'Evaluator':<35} {'Score':>6}  Result")
print(f"{'─' * 50}")
for name in evaluators:
    if name in scores:
        val, label = scores[name]
        icon = "✅" if val >= EVAL_THRESHOLD else "❌"
        print(f"{icon} {name:<33} {val:>5.1f}  {label}")
    else:
        print(f"⚠️  {name:<33}     -  no data")
print(f"{'─' * 50}")

## Step 4: Quality Gate

Check if all scores meet the threshold. In CI, this step exits non-zero to block the PR.

In [ ]:
failed = any(val < EVAL_THRESHOLD for val, _ in scores.values())
if failed:
    print(f"❌ FAILED: one or more metrics below {EVAL_THRESHOLD}")
else:
    print(f"✅ All evaluations PASSED (threshold: {EVAL_THRESHOLD})")

## Understanding MCP Role-Based Access Control

The MCP server uses a 3-layer auth pattern:

1. **AgentCore** validates the JWT signature and expiry
2. **Starlette middleware** extracts claims (roles, scopes) from the JWT payload
3. **Tool-level checks** enforce role requirements per tool

```python
TOOL_ROLES = {
    'get_stock_price': ['FinanceUser'],
    'get_employee_count': ['HRUser'],
}
```

M2M tokens (CI pipelines) have scopes but no roles → all tools accessible.
User tokens carry `custom:roles` → only matching tools are accessible.

## Next Steps

- Modify `scripts/eval_dataset.json` to add your own test prompts
- Add custom evaluators for domain-specific quality checks
- Adjust `EVAL_THRESHOLD` per environment (dev: 0.7, staging: 0.8, prod: 0.9)
- Set up the GitHub Actions workflow for automated PR gating